# 03 | Data cleaning and standardisation
**NHS A&E Intelligent Decision Support System**

**Objective:** standardise the validated source records while preserving every published count and separating national totals. The figures and decisions use the current M04 output.

## 1. Setup and methodology
The M03 validator previously checked source structure, missing fields, duplicates and national reconciliation. M04 reruns it against the current staged dataset, verifies the M01 source hashes, and then performs deterministic standardisation. A new run uses the evidence exporter to save reports, figures and test logs.

**Assumptions:** organisation code and reporting month are the intended provider grain; the reporting month is not an individual attendance date; national aggregate rows are not providers. A consistent code does not prove continuity of services.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Markdown, Image
import pandas as pd

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "src/data/data_preprocessing.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("Open the NHS-A-E-IDSS repository in VS Code first.")
sys.path.insert(0, str(repo_root))
from src.data.data_preprocessing import run_preprocessing, data_dictionary, MEASURE_NAMES
print("Project root detected. M04 will not change raw or staging data.")

## 2. Execute the validation and preprocessing pipeline
The pipeline checks input fingerprints and repeats M03 before standardising anything. It then tests row conservation, uniqueness and the preservation of the 18 numerical measures. A failed core check stops processing; an advisory is not automatically repaired. Tests are run against synthetic fixtures, not used as a substitute for validation of the NHS records.

In [ ]:
result = run_preprocessing(repo_root, run_tests=True, notebook=True)
s = result["summary"]
provider = result["provider"]
national = result["national"]
evidence = Path(result["evidence_path"])
if result["evidence_status"] != "COMPLETE":
    raise RuntimeError("Evidence run requires review. Do not use its outputs yet.")
display(Markdown(f"""### Results at a glance

| Measure | Result |
|---|---:|
| Input records | {s['input_rows']:,} |
| Provider records retained | {s['provider_rows']:,} |
| National records retained separately | {s['national_rows']:,} |
| Reporting months | {s['months']:,} |
| Distinct reported provider codes | {s['provider_codes']:,} |
| Measures conserved | {s['counts_preserved']}/18 |
| Provider records with zero Type 1 activity | {s['zero_type1_after']:,} |
| Unaccounted-for records | {s['records_unaccounted_for']} |

**Outcome:** `{s['status']}`. The evidence run is `{result['evidence_status']}`. Standardisation is not approval for forecasting; remaining M03 advisories still require investigation."""))

## 3. Separate the two levels of data
**Research question:** are any observations lost when separating the national aggregate from the provider-level data?

**Method:** classify TOTAL rows using the case- and whitespace-tolerant M03 rule. Keep all provider rows in one table and all national totals in another. The horizontal bars compare the record groups, not patient counts.

In [ ]:
display(Image(filename=str(evidence / "figures/01_record_conservation.png"), width=850))
display(Markdown(f"""**Finding:** {s['provider_rows']:,} provider records and {s['national_rows']:,} national records account for all {s['input_rows']:,} staged records.

**Interpretation:** the two populations have different aggregation levels. Combining them in a provider model would duplicate national activity.

**Decision:** use the provider table for subsequent longitudinal preparation; retain the national table for reconciliation and national-level benchmarking."""))

## 4. Standardise the schema and organisation identifiers
**Research question:** can we obtain consistent field names and a reliable provider-month key without merging organisational identities?

**Method:** rename the 22 original fields, retain the original period text, create a typed month-start date, trim outer whitespace and uppercase the organisation code. Provider name variants are retained. All four M02 provenance fields remain in the output.

In [ ]:
display(Image(filename=str(evidence / "figures/02_identifier_standardisation.png"), width=850))
display(result['changes'].rename(columns={'field':'Field','rows_standardised':'Changed rows','operation':'Operation'}))
display(provider[['period','org_code','org_name','type1_attendances']].head(6))
display(Markdown("""**Interpretation:** formatting changes help prevent false duplicates, but they do not demonstrate that differently named or coded services are equivalent. The chart counts changes across both source row groups; it is not a count of reorganisations.

**Decision:** investigate provider names and reporting continuity in M05. Keep organisation codes as identifiers rather than numeric predictors."""))

## 5. Check numerical preservation
**Research question:** did conversion from original strings to integer counts introduce any changes in published activity?

**Method:** require non-negative integer representations and verify the exact sums of all 18 measures separately for provider and national records. National totals are also reconciled within each month. Differences should be zero. No scaling, clipping, rounding or filling is applied.

In [ ]:
display(result['conservation'][['measure','provider_difference','national_difference','status']])
display(Markdown(f"""**Finding:** {s['counts_preserved']} of 18 measures are conserved across both record groups. Fresh M03 reconciliation matched {s['m03_checks']['reconciliations_matched']:,} measure-month comparisons.

**Decision:** retain all published measures in the cleaned tables. Feature selection and leakage checks are separate modelling activities."""))

## 6. Preserve reported zeros
**Research question:** did cleaning accidentally delete valid zero Type 1 activity or turn missing observations into zero?

**Method:** compare zero counts before and after standardisation. Plot the monthly share of provider records with reported zero Type 1 attendances; national totals are excluded from the denominator.

In [ ]:
display(Image(filename=str(evidence / "figures/03_zero_type1_preserved.png"), width=850))
share=provider.groupby('period')['type1_attendances'].apply(lambda v: 100*v.eq(0).mean())
display(Markdown(f"""**Finding:** {s['zero_type1_before']:,} zero observations before M04 and {s['zero_type1_after']:,} afterwards; the monthly share ranges from {share.min():.1f}% to {share.max():.1f}%.

**Interpretation:** zero is a reported value; an absent provider-month is a different condition. Persistent zero activity may reflect different service types, but this does not prove eligibility or ineligibility for any particular organisation.

**Decision:** preserve zero records at M04. In M05/M07, assess eligibility using service evidence and information available at the forecasting origin, without using future activity to select the evaluation population."""))

## 7. Data dictionary and evidence-based decisions
The dictionary describes the standardised fields and their original source names. Parquet preserves the intended types when PyArrow is installed; CSV fields must be loaded using explicit types in later stages.

In [ ]:
display(data_dictionary())
display(result['decisions'])
display(Markdown(f"""### 8. Handover to M05

The standardised tables are in `{s['output_directory']}`.

The evidence is in `{evidence.relative_to(repo_root).as_posix()}`. The directory includes the current M03 revalidation, M04 summary, source and output fingerprints, data dictionary, figures and test log. Save this executed notebook separately.

**Next:** review the {s['m03_checks']['providers_with_internal_month_gaps']} provider codes with internal reporting gaps, {s['m03_checks']['providers_with_name_variants']} codes with name variants, and {s['m03_checks']['type1_change_review_candidates']} large-change candidates from M03. Do not assume gaps are zero or automatically remove unusual demand; assess historical comparability first."""))